# WSCC 9-bus DP to EMT co-simulation: interface agreement

This notebook analyses the output of the WSCC 9-bus SFA to EMT (DP to EMT)
co-simulation. Two DPsim processes run side by side, coupled at bus 6:

- The DP follower runs the full 9-bus network with 4th-order machines. It is
  the current-source side of the interface: it receives the bus-6 current and
  exports the bus-6 voltage phasor.
- The EMT leader runs a small 3-phase sub-network at bus 6. It is the
  voltage-source side: it receives the bus-6 voltage, reconstructs the carrier,
  and exports the resulting branch current.

A VILLAS bridge demodulates and remodulates between the DP phasor and the EMT
carrier. If the coupling is correct, reconstructing the DP voltage phasor as an
EMT waveform reproduces the EMT side, and the two magnitudes agree.


The data comes from the launch script `configs/run-emt-dp-cosim.sh`, which
writes the leader and follower CSVs under its run directory. Set
`DPSIM_COSIM_LOG_DIR` to that `logs` directory, or leave the default below.


In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from villas.dataprocessing.readtools import read_timeseries_csv
from villas.dataprocessing.timeseries import TimeSeries

log_dir = Path(os.environ.get("DPSIM_COSIM_LOG_DIR", "../../../cosim-run/logs"))
leader_csv = (
    log_dir / "EMT_3Ph_9bus_load6_cosim_leader" / "EMT_3Ph_9bus_load6_cosim_leader.csv"
)
follower_csv = log_dir / "DP_Ph1_9bus_4order_cosim_follower.csv"

# Frequency the bridge uses to demodulate and remodulate the interface signals.
f0 = 50.0

In [ ]:
leader = read_timeseries_csv(str(leader_csv), print_status=False)
follower = read_timeseries_csv(str(follower_csv), print_status=False)


def trim(series):
    # The real-time logger preallocates a couple of trailing rows that stay at
    # time zero; keep only the monotonic prefix and align all series in length.
    lengths = []
    for ts in series.values():
        lengths.append(int(np.argmax(ts.time)) + 1)
    n = min(lengths)
    return {
        name: TimeSeries(name, ts.time[:n], ts.values[:n])
        for name, ts in series.items()
    }


leader = trim(leader)
follower = trim(follower)
n = min(len(leader["ax"].time), len(follower["BUS6"].time))
leader = {k: TimeSeries(k, v.time[:n], v.values[:n]) for k, v in leader.items()}
follower = {k: TimeSeries(k, v.time[:n], v.values[:n]) for k, v in follower.items()}
t = leader["ax"].time
print(f"{n} samples, {t[0]:.3g} to {t[-1]:.3g} s")

## Reconstructing the DP voltage as an EMT waveform

The DP follower exports the bus-6 voltage as one positive-sequence phasor.
`create_emt_from_dp` shifts that phasor back to the carrier frequency, giving
the phase-A EMT waveform. It should sit on the voltage the EMT leader measured
at bus 6 (a) and on the reference the bridge handed it (ax).


In [ ]:
emt_recon = TimeSeries.create_emt_from_dp(
    [follower["BUS6"]], [f0], "bus6_reconstructed"
)

# Normalized RMSE against the EMT reference, over the settled window (t > 2 s).
mask = t > 2.0
resid = emt_recon.values[mask] - leader["ax"].values[mask]
nrmse = np.sqrt(np.mean(resid**2)) / np.sqrt(np.mean(leader["ax"].values[mask] ** 2))
print(f"normalized RMSE, reconstructed vs leader ax: {100 * nrmse:.2f} %")

win = (t >= 6.0) & (t <= 6.06)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t[win], leader["a"].values[win] / 1e3, label="EMT bus-6 voltage a")
ax.plot(t[win], emt_recon.values[win] / 1e3, "--", label="reconstructed from DP phasor")
ax.set_xlabel("time [s]")
ax.set_ylabel("voltage [kV]")
ax.set_title("Bus-6 voltage: EMT measurement vs DP phasor reconstruction")
ax.legend(loc="upper right")
ax.grid(True)
plt.tight_layout()

## Voltage magnitude agreement

The magnitude of the DP phasor should equal the collective magnitude of the
three EMT phases, sqrt(2/3 (ax^2 + bx^2 + cx^2)), at every instant. This is
phase independent, so it isolates the amplitude match from any carrier phase.


In [ ]:
dp_v_mag = follower["BUS6"].abs()
emt_v_env = TimeSeries(
    "emt_envelope",
    t,
    np.sqrt(
        2.0
        / 3.0
        * (
            leader["ax"].values ** 2
            + leader["bx"].values ** 2
            + leader["cx"].values ** 2
        )
    ),
)

mask = t > 2.0
rel = (
    np.abs(emt_v_env.values[mask] - dp_v_mag.values[mask]) / dp_v_mag.values[mask]
).mean()
print(f"mean relative error (t > 2 s): {100 * rel:.4f} %")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, dp_v_mag.values / 1e3, label="DP bus-6 phasor magnitude")
ax.plot(t, emt_v_env.values / 1e3, "--", label="EMT reconstructed envelope")
ax.set_xlabel("time [s]")
ax.set_ylabel("bus-6 voltage [kV]")
ax.set_title("Interface voltage magnitude: DP phasor vs EMT envelope")
ax.legend()
ax.grid(True)
plt.tight_layout()

## Current at the interface

The EMT side exports its phase-A branch current. The bridge demodulates it into
the phasor the DP current source injects at bus 6, so the EMT phase-A current
oscillates within the envelope set by the DP current phasor magnitude.


In [ ]:
dp_i_mag = follower["BUS6_i"].abs()
win = (t >= 6.0) & (t <= 6.06)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t[win], leader["a_i"].values[win], label="EMT phase-A current a_i")
ax.plot(t[win], dp_i_mag.values[win], "k--", label="DP current phasor magnitude")
ax.plot(t[win], -dp_i_mag.values[win], "k--")
ax.set_xlabel("time [s]")
ax.set_ylabel("current [A]")
ax.set_title("Interface current: EMT phase-A vs DP phasor envelope")
ax.legend(loc="upper right")
ax.grid(True)
plt.tight_layout()

## Generator response

The DP follower carries the machine dynamics. The rotor speeds stay close to
one per unit, confirming the coupled system runs stably over the interface.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for gen in ["GEN1", "GEN2", "GEN3"]:
    ax.plot(t, follower[f"{gen}.omega"].values, label=gen)
ax.set_xlabel("time [s]")
ax.set_ylabel("rotor speed [p.u.]")
ax.set_title("Generator rotor speeds (DP follower)")
ax.legend()
ax.grid(True)
plt.tight_layout()

The reconstructed DP voltage tracks the EMT measurement and the magnitudes
agree to a small fraction of a percent, so the two domains agree at bus 6.

This example uses a balanced, positive-sequence coupling: the single DP phasor
is fanned out to three balanced phases, and only the positive-sequence current
is fed back. A three-phase DP interface is a planned extension. The bridge
demodulates at a fixed 50 Hz window while the network runs at 60 Hz; making the
coupling frequency track the system frequency is a follow-up.
